In [ ]:
import pandas as pd
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

gdf = gpd.read_file('Regions_no_province_boundaries.json')
gdf['HealthRegion'] = gdf['HealthRegion'].astype(int)

# หาพิกัดจุดกึ่งกลาง (Centroid) ของแต่ละเขตสุขภาพ
gdf['lat'] = gdf.geometry.centroid.y
gdf['lon'] = gdf.geometry.centroid.x

# เรียงคอลัมน์: Location_ID, Latitude, Longitude (SaTScan ไม่เอา Header)
geo_df = gdf[['HealthRegion', 'lat', 'lon']].sort_values('HealthRegion')
geo_df.to_csv('SaTScan_Coordinates.geo', index=False, header=False, sep=',')
print("✅ สร้างไฟล์พิกัดสำเร็จ: SaTScan_Coordinates.geo")

In [ ]:
import pandas as pd

# โหลดข้อมูล
df = pd.read_csv('/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/MDR/spatiotemperal/aba_province/acinetobacter_baumannii.csv')
df_map = pd.read_csv('/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/notebooks/provinces.csv')
# กำหนดชื่อ Pattern ทั้ง 5 ตามที่ปรากฏในข้อมูลจริง (CEPHEMS = Third- and fourth-generation cephalosporins)
top_patterns = [
    "AMINOGLYCOSIDES, CARBAPENEMS, CEPHEMS, FLUOROQUINOLONES, FOLATE PATHWAY ANTAGONISTS, β-LACTAM COMBINATION AGENTS",
    "AMINOGLYCOSIDES, CARBAPENEMS, CEPHEMS, FLUOROQUINOLONES, β-LACTAM COMBINATION AGENTS",
    "CARBAPENEMS, CEPHEMS, FLUOROQUINOLONES, FOLATE PATHWAY ANTAGONISTS, β-LACTAM COMBINATION AGENTS",
    "CARBAPENEMS, CEPHEMS, FLUOROQUINOLONES, β-LACTAM COMBINATION AGENTS",
    "CARBAPENEMS, CEPHEMS, FOLATE PATHWAY ANTAGONISTS, β-LACTAM COMBINATION AGENTS"
]

prov_eng_mapping = df_map.drop_duplicates(subset=['province_name']).set_index('province_name')['province_english'].to_dict()
    
    # อัปเดตคอลัมน์ province ใน df หลักให้เป็นชื่อภาษาอังกฤษ
    # (สมมติว่าใน df เดิม คอลัมน์ province มีชื่อจังหวัดเป็นภาษาไทยอยู่แล้ว)
df['province'] = df['province'].map(prov_eng_mapping)

# สร้างคอลัมน์วันที่ให้อยู่ในรูปแบบ YYYY/MM ที่ SaTScan อ่านได้ง่าย
df['date'] = df['year'].astype(str) + '/' + df['month'].astype(str).str.zfill(2)

pop_df = df[['province', 'date', 'total_rows_in_month']].drop_duplicates()
pop_df.to_csv('province_population.pop', index=False, header=False)
print(f"Created population.pop with {len(pop_df)} rows")

# 2. สร้างไฟล์ Case (.cas) สำหรับแต่ละ Pattern
for i, pattern in enumerate(top_patterns, start=1):
    case_df = df[df['Resistant_Drug_Classes'] == pattern][['province', 'pattern_count', 'date']]
    case_df.to_csv(f'province_pattern_{i}.cas', index=False, header=False)
    print(f"Created province_pattern_{i}.cas with {len(case_df)} rows for Pattern {i}")

Created population.pop with 5090 rows
Created province_pattern_1.cas with 4981 rows for Pattern 1
Created province_pattern_2.cas with 3943 rows for Pattern 2
Created province_pattern_3.cas with 3049 rows for Pattern 3
Created province_pattern_4.cas with 2954 rows for Pattern 4
Created province_pattern_5.cas with 1072 rows for Pattern 5


In [9]:
import pandas as pd
df = pd.read_csv('/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/data/MDR/AllYears_MDR_Pattern.csv') 

/var/folders/f0/09md1vw56wn4jjhchxblsy0r0000gn/T/ipykernel_57303/4287017567.py:2: DtypeWarning: Columns (2,7,13,19,25,31,40,42,44,45,46,47,49,51,55,59,60,61,62,64,65,68,69,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,97) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/data/MDR/AllYears_MDR_Pattern.csv')


In [12]:
df['laboratory'].unique()

array(['LPA', 'MKH', 'NHL', 'PEB', 'TKS', 'SRT', 'BRH', 'PYY', 'KKH',
       'RYH', 'SBR', 'NKP', 'KJB', 'CPL', 'PTH', 'CHN', 'PBR', 'UDH',
       'MGH', 'THU', 'NPM', 'PGN', 'RAE', 'PAH', 'SKN', 'PYO', 'SRB',
       'NSI', 'PUK', 'LUR', 'TRA', 'SRI', 'UTD', 'MUK', 'SKO', 'PPK',
       'RAT', 'UTN', 'BHU', 'HAT', 'SNK', 'PIJ', 'YAL', 'SKH', 'SAT',
       'BUD', 'LBH', 'NBP', 'NAR', 'NAH', 'BKH', 'NKY', 'KPP', 'BAM',
       'SKE', 'SLH', 'TMH', 'PCK', 'ANJ', 'SSK', 'CHH', 'PTN', 'CBR',
       'CRH', 'SKT', 'CPH', 'PKH', 'PHR', 'RAN', 'AYH', 'YAS', 'NKH',
       'TKP', 'SAP', 'P2H', 'VAJ', 'NRH', 'KRS', 'NUH', 'SSH', 'ATH',
       'SRH', 'PRI', 'CKH', 'LSH', 'MSH', 'LPN', 'RSM', 'SKR', 'SSW',
       'TRH', 'KRB', 'NOP', 'LED', 'PRH', 'HUA', 'WIC', 'BAN', 'NPH',
       'CMH', 'SPK', 'SPR', 'H21', 'FNG', 'CTH', 'QSN', 'CHP', 'SUT',
       'TBH', 'KLH', 'AMH', 'ANM', 'RAM', 'SKL', 'KPH', 'NTW', 'SOM',
       'BPH', 'BNS', 'NIH', 'KTB', 'KRH', 'TRC', 'PTB'], dtype=object)

In [ ]:
import pandas as pd
import os

# 1. กำหนด Path ของไฟล์ (ปรับแก้ตามโฟลเดอร์จริงในเครื่องคุณได้เลยครับ)
INPUT_PATH = '/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/notebooks/provinces.csv'
OUTPUT_DIR = '/Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/notebooks'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. โหลดไฟล์พิกัดจังหวัด
df = pd.read_csv(INPUT_PATH)

# 3. เลือกเฉพาะคอลัมน์ที่ SaTScan ต้องการ: [รหัสพื้นที่ (ชื่อจังหวัด), ละติจูด, ลองจิจูด]
geo_df = df[['province_english', 'province_lat', 'province_lon']]

# 4. บันทึกเป็นไฟล์ .geo (โดยไม่เอาหัวตาราง และไม่เอาลำดับ index)
output_file = os.path.join(OUTPUT_DIR, 'SaTScan_Province_Coordinates.geo')
geo_df.to_csv(output_file, index=False, header=False)

print(f"✅ สร้างไฟล์พิกัดสำหรับ SaTScan ระดับจังหวัดเสร็จสมบูรณ์!")
print(f"ไฟล์ถูกบันทึกไว้ที่: {output_file}")

✅ สร้างไฟล์พิกัดสำหรับ SaTScan ระดับจังหวัดเสร็จสมบูรณ์!
ไฟล์ถูกบันทึกไว้ที่: /Users/chanokchonkarinrak/Documents/GitHub/AMR_Thesis/notebooks/SaTScan_Province_Coordinates.geo
